
# EDA Notebook 2 v2 — SemEval-2014 Task 4 (Restaurants and Laptops)

**Thesis:** *An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns*  
**Student:** Abhishek Tomar | PN1196973 | LJMU

## Purpose of this notebook

This notebook audits SemEval-2014 **after** the TweetEval linguistic-tagging rules have been frozen.

SemEval-2014 is **not** treated as a fourth training dataset and is **not** used for model or feature selection. Its role is restricted to an **external domain/task-shift stress test** because its labels are attached to aspect–sentiment pairs rather than ordinary sentence-level sentiment.

The notebook therefore:

1. loads the laptop and restaurant domains;
2. audits row-level and unique-sentence structure;
3. quantifies repeated sentences, multi-aspect annotations and conflicting sentiment labels;
4. checks train/test sentence overlap;
5. loads the frozen TweetEval tagging configuration without modifying it;
6. applies the exact same linguistic-tagging rules to raw SemEval text;
7. reports overlap before exclusive assignment;
8. reports subgroup coverage at both row and unique-sentence level;
9. determines which external subgroup comparisons, if any, meet the pre-specified `N >= 30` interpretation threshold;
10. saves reproducible audit outputs.

**Important:** subgroup scarcity does not trigger threshold adjustment.


## Cell 1 — Install and import libraries

In [2]:

!pip install -q datasets emoji pyarrow

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import Counter
import glob
import json
import re

import numpy as np
import pandas as pd
import emoji
from datasets import load_dataset

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

print("Libraries loaded.")


Mounted at /content/drive
Libraries loaded.


## Cell 2 — Locate and load the frozen TweetEval tagging specification

In [3]:

CONFIG_NAME = "tweeteval_freeze_config.json"

matches = glob.glob(
    f"/content/drive/MyDrive/**/{CONFIG_NAME}",
    recursive=True
)

if not matches:
    raise FileNotFoundError(
        f"Could not find {CONFIG_NAME} anywhere under /content/drive/MyDrive. "
        "Run EDA_01_TweetEval_FINAL_v2 first."
    )

# Prefer the copy inside Results/TweetEval if more than one exists.
preferred = [p for p in matches if "/Results/TweetEval/" in p.replace("\\", "/")]
config_path = Path(preferred[0] if preferred else matches[0])

if len(matches) > 1:
    print("Multiple freeze-config copies found.")
    print("Using:", config_path)
else:
    print("Freeze config:", config_path)

with open(config_path, "r", encoding="utf-8") as f:
    freeze = json.load(f)

# Reproducibility assertions: fail loudly rather than drift silently.
assert freeze["emoji_heavy"]["operator"] == ">"
assert float(freeze["emoji_heavy"]["threshold"]) == 0.05
assert freeze["slang_heavy"]["operator"] == ">"
assert float(freeze["slang_heavy"]["threshold"]) == 0.10
assert int(freeze["slang_heavy"]["lexicon_size"]) == 51
assert int(freeze["reference_unmarked"]["minimum_word_count"]) == 8
assert freeze["priority_rule"] == [
    "sarcasm-indicated",
    "emoji-heavy",
    "slang-heavy",
    "reference-unmarked",
    "other",
]
assert int(freeze["minimum_n_for_stable_quantitative_interpretation"]) == 30

RESULTS_ROOT = config_path.parent.parent
OUTDIR = RESULTS_ROOT / "SemEval2014"
OUTDIR.mkdir(parents=True, exist_ok=True)

print("Frozen thresholds:")
print(" emoji-heavy >", freeze["emoji_heavy"]["threshold"])
print(" slang-heavy >", freeze["slang_heavy"]["threshold"])
print(" reference/unmarked minimum words =", freeze["reference_unmarked"]["minimum_word_count"])
print(" interpretation N >=", freeze["minimum_n_for_stable_quantitative_interpretation"])
print("Output directory:", OUTDIR)


Freeze config: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/TweetEval/tweeteval_freeze_config.json
Frozen thresholds:
 emoji-heavy > 0.05
 slang-heavy > 0.1
 reference/unmarked minimum words = 8
 interpretation N >= 30
Output directory: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/SemEval2014


## Cell 3 — Load SemEval-2014 and preserve aspect-level structure

In [4]:

DATASETS = {
    "laptop": "yqzheng/semeval2014_laptops",
    "restaurant": "yqzheng/semeval2014_restaurants",
}
LABEL_MAP = {-1: "negative", 0: "neutral", 1: "positive"}

rows = []
skipped_labels = []

for domain, repo in DATASETS.items():
    ds = load_dataset(repo)
    for split_name, split_data in ds.items():
        kept = 0
        skipped = 0
        for row_id, item in enumerate(split_data):
            raw_label = item["label"]
            sentiment = LABEL_MAP.get(raw_label)
            if sentiment is None:
                skipped += 1
                skipped_labels.append({
                    "domain": domain,
                    "split": split_name,
                    "row_id": row_id,
                    "raw_label": raw_label,
                })
                continue

            rows.append({
                "domain": domain,
                "split": split_name,
                "source_row_id": row_id,
                "text": item["text"],
                "aspect": item["aspect"],
                "sentiment": sentiment,
                "raw_label": raw_label,
            })
            kept += 1

        print(f"{domain:10s} {split_name:5s}: kept={kept:,}, skipped={skipped:,}")

df = pd.DataFrame(rows)
df["text"] = df["text"].fillna("").astype(str)
df["aspect"] = df["aspect"].fillna("").astype(str)
df["sentiment"] = pd.Categorical(
    df["sentiment"],
    categories=["negative", "neutral", "positive"],
    ordered=True
)

print("\nRows loaded:", f"{len(df):,}")
print("Columns:", list(df.columns))
print("Skipped unsupported labels:", len(skipped_labels))
df.head(3)


README.md:   0%|          | 0.00/672 [00:00<?, ?B/s]

data/train-00000-of-00001-0a88592fd0965e(…): reconstructing file:   0%|          |  0.00B /  125kB            

data/train-00000-of-00001-0a88592fd0965e(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-7e586727cd016f7(…): reconstructing file:   0%|          |  0.00B / 32.4kB            

data/test-00000-of-00001-7e586727cd016f7(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-c5b7c85e409aed4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3608 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1120 [00:00<?, ? examples/s]

restaurant train: kept=3,608, skipped=0
restaurant test : kept=1,120, skipped=0

Rows loaded: 7,694
Columns: ['domain', 'split', 'source_row_id', 'text', 'aspect', 'sentiment', 'raw_label']
Skipped unsupported labels: 0


,domain,split,source_row_id,text,aspect,sentiment,raw_label
0,laptop,train,0,I charge it at night and skip taking the cord ...,cord,neutral,0
1,laptop,train,1,I charge it at night and skip taking the cord ...,battery life,positive,1
2,laptop,train,2,The tech guy then said the service center does...,service center,negative,-1


## Cell 4 — Integrity audit: missing values, empty text, row counts and unique sentences

In [5]:

def normalise_for_duplicate_audit(text):
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

df["text_norm"] = df["text"].map(normalise_for_duplicate_audit)

missing_summary = df.isna().sum().to_dict()
empty_text_n = int((df["text"].str.strip() == "").sum())
empty_aspect_n = int((df["aspect"].str.strip() == "").sum())

row_counts = (
    df.groupby(["domain", "split"], observed=True)
      .size()
      .rename("aspect_rows")
      .reset_index()
)

sentence_counts = (
    df.groupby(["domain", "split"], observed=True)["text_norm"]
      .nunique()
      .rename("unique_sentences")
      .reset_index()
)

integrity_counts = row_counts.merge(sentence_counts, on=["domain", "split"], how="outer")

print("Missing values:", missing_summary)
print("Empty text rows:", empty_text_n)
print("Empty aspect rows:", empty_aspect_n)
print("\nRows and unique sentences:")
print(integrity_counts.to_string(index=False))

print("\nOverall aspect rows:", f"{len(df):,}")
print("Overall unique sentence texts:", f"{df['text_norm'].nunique():,}")


Missing values: {'domain': 0, 'split': 0, 'source_row_id': 0, 'text': 0, 'aspect': 0, 'sentiment': 0, 'raw_label': 0, 'text_norm': 0}
Empty text rows: 0
Empty aspect rows: 0

Rows and unique sentences:
    domain split  aspect_rows  unique_sentences
    laptop  test          638               411
    laptop train         2328              1457
restaurant  test         1120               600
restaurant train         3608              1978

Overall aspect rows: 7,694
Overall unique sentence texts: 4,444


## Cell 5 — Multi-aspect structure and conflicting sentence labels

In [6]:

sentence_audit = (
    df.groupby(["domain", "split", "text_norm"], observed=True)
      .agg(
          aspect_rows=("aspect", "size"),
          unique_aspects=("aspect", "nunique"),
          unique_sentiments=("sentiment", "nunique"),
          sentiments=("sentiment", lambda x: "|".join(sorted(set(map(str, x))))),
          aspects=("aspect", lambda x: " | ".join(map(str, x))),
          example_text=("text", "first"),
      )
      .reset_index()
)

sentence_audit["multi_aspect"] = sentence_audit["aspect_rows"] > 1
sentence_audit["conflicting_sentiment"] = sentence_audit["unique_sentiments"] > 1

multi_n = int(sentence_audit["multi_aspect"].sum())
conflict_n = int(sentence_audit["conflicting_sentiment"].sum())

print("Unique domain/split sentence records:", f"{len(sentence_audit):,}")
print("Sentences with >1 aspect row:", f"{multi_n:,}",
      f"({multi_n/len(sentence_audit)*100:.2f}%)")
print("Sentences with conflicting sentiment labels:", f"{conflict_n:,}",
      f"({conflict_n/len(sentence_audit)*100:.2f}%)")

conflicts = sentence_audit[sentence_audit["conflicting_sentiment"]].copy()
conflicts.head(10)


Unique domain/split sentence records: 4,446
Sentences with >1 aspect row: 1,985 (44.65%)
Sentences with conflicting sentiment labels: 604 (13.59%)


,domain,split,text_norm,aspect_rows,unique_aspects,unique_sentiments,sentiments,aspects,example_text,multi_aspect,conflicting_sentiment
8,laptop,test,a veryimportant feature is firewire 800 which ...,5,4,3,negative|neutral|positive,Firewire 800 | USB3 | USB3 | MAC OS | Windows 7,A veryimportant feature is Firewire 800 which ...,True,True
9,laptop,test,"additionally, there is barely a ventilation sy...",3,3,2,negative|neutral,ventilation system | watching videos | playing...,"Additionally, there is barely a ventilation sy...",True,True
12,laptop,test,after replacing the spinning hard disk with an...,2,2,2,neutral|positive,spinning hard disk | ssd drive,After replacing the spinning hard disk with an...,True,True
13,laptop,test,air has higher resolution but the fonts are sm...,2,2,2,negative|positive,resolution | fonts,air has higher resolution but the fonts are sm...,True,True
16,laptop,test,"also, in using the built-in camera, my voice r...",2,2,2,negative|neutral,built-in camera | voice recording,"Also, in using the built-in camera, my voice r...",True,True
19,laptop,test,and as for all the fancy finger swipes -- i ju...,2,2,2,negative|neutral,finger swipes | mouse,And as for all the fancy finger swipes -- I ju...,True,True
31,laptop,test,apple removed the dvd drive firewire port (wil...,3,3,2,negative|neutral,DVD drive Firewire port | adapter | SDXC slot,Apple removed the DVD drive Firewire port (wil...,True,True
65,laptop,test,"fast, great screen, beautiful apps for a lapto...",4,4,2,neutral|positive,screen | apps | priced | cost,"fast, great screen, beautiful apps for a lapto...",True,True
69,laptop,test,from the speed to the multi touch gestures thi...,4,4,2,negative|positive,operating system | Windows | speed | multi tou...,From the speed to the multi touch gestures thi...,True,True
76,laptop,test,has all the other features i wanted including ...,5,5,2,neutral|positive,features | VGA port | HDMI | ethernet | USB ports,Has all the other features I wanted including ...,True,True


## Cell 6 — Cross-split sentence overlap audit

In [7]:

# Within-domain train/test overlap
cross_split = (
    df.groupby(["domain", "text_norm"], observed=True)
      .agg(
          split_count=("split", "nunique"),
          splits=("split", lambda x: "|".join(sorted(set(map(str, x))))),
          sentiment_count=("sentiment", "nunique"),
          sentiments=("sentiment", lambda x: "|".join(sorted(set(map(str, x))))),
          example_text=("text", "first"),
      )
      .reset_index()
)

cross_split = cross_split[cross_split["split_count"] > 1].copy()

# Cross-domain repeated sentence text is recorded separately.
cross_domain = (
    df.groupby("text_norm", observed=True)
      .agg(
          domain_count=("domain", "nunique"),
          domains=("domain", lambda x: "|".join(sorted(set(map(str, x))))),
          split_count=("split", "nunique"),
          example_text=("text", "first"),
      )
      .reset_index()
)
cross_domain = cross_domain[cross_domain["domain_count"] > 1].copy()

print("Within-domain sentence groups appearing in >1 split:", len(cross_split))
print("Sentence groups appearing in >1 domain:", len(cross_domain))

if len(cross_split):
    display(cross_split.head(20))


Within-domain sentence groups appearing in >1 split: 2
Sentence groups appearing in >1 domain: 0


,domain,text_norm,split_count,splits,sentiment_count,sentiments,example_text
751,laptop,if you want more information on macs i suggest...,2,test|train,1,neutral,If you want more information on macs I suggest...
2679,restaurant,i would definitely go back -- if only for some...,2,test|train,1,positive,I would definitely go back -- if only for some...


## Cell 7 — Frozen linguistic-tagging functions

In [8]:

SLANG = list(freeze["slang_heavy"]["lexicon"])
SLANG_SET = set(SLANG)
SARCASTIC_RE = re.compile(
    freeze["sarcasm_indicated"]["regex"],
    flags=re.IGNORECASE,
)

EMOJI_THRESHOLD = float(freeze["emoji_heavy"]["threshold"])
SLANG_THRESHOLD = float(freeze["slang_heavy"]["threshold"])
REF_MIN_WORDS = int(freeze["reference_unmarked"]["minimum_word_count"])
MIN_STABLE_N = int(freeze["minimum_n_for_stable_quantitative_interpretation"])

# The slang lexicon consists of single-token forms, so token-level matching is exact.
SLANG_TOKEN_RE = re.compile(
    r"(?<!\w)(" + "|".join(sorted(map(re.escape, SLANG), key=len, reverse=True)) + r")(?!\w)",
    flags=re.IGNORECASE,
)

def word_count_raw(text):
    return len(str(text).split())

def emoji_count_full_sequences(text):
    return len(emoji.emoji_list(str(text)))

def slang_count(text):
    return len(SLANG_TOKEN_RE.findall(str(text).lower()))

def sarcasm_indicated(text):
    return bool(SARCASTIC_RE.search(str(text).lower()))

def apply_frozen_flags(frame):
    out = frame.copy()

    out["word_count"] = out["text"].map(word_count_raw)
    out["emoji_count"] = out["text"].map(emoji_count_full_sequences)
    out["slang_count"] = out["text"].map(slang_count)

    denom = out["word_count"].clip(lower=1)
    out["emoji_density"] = out["emoji_count"] / denom
    out["slang_ratio"] = out["slang_count"] / denom

    out["is_emoji_heavy"] = out["emoji_density"] > EMOJI_THRESHOLD
    out["is_slang_heavy"] = out["slang_ratio"] > SLANG_THRESHOLD
    out["is_sarcasm_indicated"] = out["text"].map(sarcasm_indicated)

    out["is_reference_candidate"] = (
        (~out["is_emoji_heavy"])
        & (~out["is_slang_heavy"])
        & (~out["is_sarcasm_indicated"])
        & (out["word_count"] >= REF_MIN_WORDS)
    )

    def assign(row):
        if row["is_sarcasm_indicated"]:
            return "sarcasm-indicated"
        if row["is_emoji_heavy"]:
            return "emoji-heavy"
        if row["is_slang_heavy"]:
            return "slang-heavy"
        if row["is_reference_candidate"]:
            return "reference-unmarked"
        return "other"

    out["subgroup"] = out.apply(assign, axis=1)
    return out

df = apply_frozen_flags(df)

print("Frozen linguistic rules applied without threshold modification.")


Frozen linguistic rules applied without threshold modification.


## Cell 8 — Pre-priority flag prevalence and overlap

In [9]:

FLAGS = [
    "is_emoji_heavy",
    "is_slang_heavy",
    "is_sarcasm_indicated",
    "is_reference_candidate",
]

flag_prevalence = pd.DataFrame({
    "flag": FLAGS,
    "count": [int(df[c].sum()) for c in FLAGS],
    "percentage": [float(df[c].mean() * 100) for c in FLAGS],
})

print("Independent pre-priority flags:")
print(flag_prevalence.to_string(index=False))

focus_flags = ["is_emoji_heavy", "is_slang_heavy", "is_sarcasm_indicated"]
overlap_matrix = pd.DataFrame(
    index=focus_flags,
    columns=focus_flags,
    dtype=int,
)

for a in focus_flags:
    for b in focus_flags:
        overlap_matrix.loc[a, b] = int((df[a] & df[b]).sum())

def overlap_pattern(row):
    labels = []
    if row["is_emoji_heavy"]:
        labels.append("emoji-heavy")
    if row["is_slang_heavy"]:
        labels.append("slang-heavy")
    if row["is_sarcasm_indicated"]:
        labels.append("sarcasm-indicated")
    return " + ".join(labels) if labels else "none"

df["pre_priority_pattern"] = df.apply(overlap_pattern, axis=1)
overlap_patterns = (
    df["pre_priority_pattern"]
      .value_counts()
      .rename_axis("pattern")
      .reset_index(name="count")
)

print("\nOverlap matrix:")
print(overlap_matrix)
print("\nOverlap patterns:")
print(overlap_patterns.to_string(index=False))


Independent pre-priority flags:
                  flag  count  percentage
        is_emoji_heavy      0    0.000000
        is_slang_heavy      6    0.077983
  is_sarcasm_indicated     26    0.337926
is_reference_candidate   6869   89.277359

Overlap matrix:
                      is_emoji_heavy  is_slang_heavy  is_sarcasm_indicated
is_emoji_heavy                   0.0             0.0                   0.0
is_slang_heavy                   0.0             6.0                   0.0
is_sarcasm_indicated             0.0             0.0                  26.0

Overlap patterns:
          pattern  count
             none   7662
sarcasm-indicated     26
      slang-heavy      6


## Cell 9 — Final subgroup distribution by domain and split

In [10]:

SUBGROUP_ORDER = [
    "sarcasm-indicated",
    "emoji-heavy",
    "slang-heavy",
    "reference-unmarked",
    "other",
]

subgroup_rows = (
    df.groupby(["domain", "split", "subgroup"], observed=True)
      .size()
      .rename("aspect_rows")
      .reset_index()
)

subgroup_rows["subgroup"] = pd.Categorical(
    subgroup_rows["subgroup"],
    categories=SUBGROUP_ORDER,
    ordered=True,
)
subgroup_rows = subgroup_rows.sort_values(["domain", "split", "subgroup"])

print(subgroup_rows.to_string(index=False))


    domain split           subgroup  aspect_rows
    laptop  test        slang-heavy            1
    laptop  test reference-unmarked          545
    laptop  test              other           92
    laptop train  sarcasm-indicated            2
    laptop train        slang-heavy            1
    laptop train reference-unmarked         2130
    laptop train              other          195
restaurant  test  sarcasm-indicated            7
restaurant  test reference-unmarked          987
restaurant  test              other          126
restaurant train  sarcasm-indicated           17
restaurant train        slang-heavy            4
restaurant train reference-unmarked         3207
restaurant train              other          380


## Cell 10 — Unique-sentence subgroup coverage

In [11]:

# Tagging depends only on raw sentence text, so repeated aspect rows should receive the same subgroup.
consistency = (
    df.groupby(["domain", "split", "text_norm"], observed=True)["subgroup"]
      .nunique()
)
assert int((consistency > 1).sum()) == 0, "A repeated sentence received inconsistent subgroup tags."

sentence_tag = (
    df.sort_values(["domain", "split", "text_norm"])
      .drop_duplicates(["domain", "split", "text_norm"])
      [["domain", "split", "text_norm", "text", "subgroup",
        "word_count", "emoji_count", "emoji_density",
        "slang_count", "slang_ratio",
        "is_emoji_heavy", "is_slang_heavy", "is_sarcasm_indicated",
        "is_reference_candidate"]]
      .copy()
)

unique_coverage = (
    sentence_tag.groupby(["domain", "split", "subgroup"], observed=True)
      .size()
      .rename("unique_sentences")
      .reset_index()
)

unique_coverage["eligible_n_ge_30"] = unique_coverage["unique_sentences"] >= MIN_STABLE_N
unique_coverage["interpretation"] = np.where(
    unique_coverage["eligible_n_ge_30"],
    "quantitative coverage available; still subject to task-compatibility limits",
    "descriptive/exploratory only"
)

print(unique_coverage.to_string(index=False))


    domain split           subgroup  unique_sentences  eligible_n_ge_30                                                              interpretation
    laptop  test              other                82              True quantitative coverage available; still subject to task-compatibility limits
    laptop  test reference-unmarked               328              True quantitative coverage available; still subject to task-compatibility limits
    laptop  test        slang-heavy                 1             False                                                descriptive/exploratory only
    laptop train              other               170              True quantitative coverage available; still subject to task-compatibility limits
    laptop train reference-unmarked              1285              True quantitative coverage available; still subject to task-compatibility limits
    laptop train  sarcasm-indicated                 1             False                                         

## Cell 11 — Subgroup × sentiment structure at aspect-row level

In [12]:

subgroup_class = (
    df.groupby(["domain", "split", "subgroup", "sentiment"], observed=True)
      .size()
      .rename("count")
      .reset_index()
)

subgroup_class["within_subgroup_pct"] = (
    subgroup_class["count"]
    / subgroup_class.groupby(["domain", "split", "subgroup"], observed=True)["count"].transform("sum")
    * 100
)

print(subgroup_class.to_string(index=False))


    domain split           subgroup sentiment  count  within_subgroup_pct
    laptop  test              other  negative     15            16.304348
    laptop  test              other   neutral     10            10.869565
    laptop  test              other  positive     67            72.826087
    laptop  test reference-unmarked  negative    112            20.550459
    laptop  test reference-unmarked   neutral    159            29.174312
    laptop  test reference-unmarked  positive    274            50.275229
    laptop  test        slang-heavy  negative      1           100.000000
    laptop train              other  negative     61            31.282051
    laptop train              other   neutral     18             9.230769
    laptop train              other  positive    116            59.487179
    laptop train reference-unmarked  negative    808            37.934272
    laptop train reference-unmarked   neutral    446            20.938967
    laptop train reference-unmarked  p

## Cell 12 — External-role decision

In [13]:

role_decision = {
    "dataset": "SemEval-2014 Task 4",
    "domains": ["laptop", "restaurant"],
    "source_task": "aspect-based sentiment analysis",
    "role_in_thesis": "domain_task_shift_stress_test",
    "used_for_training": False,
    "used_for_model_selection": False,
    "used_for_feature_selection": False,
    "clean_same_task_external_validation": False,
    "reason": (
        "Labels are attached to aspect-sentiment pairs; the same sentence can carry "
        "multiple aspects and conflicting sentiment labels. External results therefore "
        "reflect both domain shift and task/label-unit shift."
    ),
    "linguistic_tagging": {
        "rule_source": "TweetEval frozen configuration",
        "threshold_tuning_on_semeval": False,
        "minimum_n_for_stable_quantitative_interpretation": MIN_STABLE_N,
        "coverage_unit_for_eligibility": "unique sentence text within domain/split",
    },
    "interpretation_constraint": (
        "Subgroup comparisons are reported only when coverage is sufficient and the "
        "task mismatch is explicitly acknowledged; absence of coverage is not repaired "
        "by changing thresholds."
    ),
}

print(json.dumps(role_decision, indent=2))


{
  "dataset": "SemEval-2014 Task 4",
  "domains": [
    "laptop",
    "restaurant"
  ],
  "source_task": "aspect-based sentiment analysis",
  "role_in_thesis": "domain_task_shift_stress_test",
  "used_for_training": false,
  "used_for_model_selection": false,
  "used_for_feature_selection": false,
  "clean_same_task_external_validation": false,
  "reason": "Labels are attached to aspect-sentiment pairs; the same sentence can carry multiple aspects and conflicting sentiment labels. External results therefore reflect both domain shift and task/label-unit shift.",
  "linguistic_tagging": {
    "rule_source": "TweetEval frozen configuration",
    "threshold_tuning_on_semeval": false,
    "minimum_n_for_stable_quantitative_interpretation": 30,
    "coverage_unit_for_eligibility": "unique sentence text within domain/split"
  },
  "interpretation_constraint": "Subgroup comparisons are reported only when coverage is sufficient and the task mismatch is explicitly acknowledged; absence of cover

## Cell 13 — Save authoritative SemEval audit outputs

In [14]:

# Save main tagged dataframe
df.to_parquet(OUTDIR / "df_semeval2014_final.parquet", index=False)

# Core tables
integrity_counts.to_csv(OUTDIR / "semeval_integrity_counts.csv", index=False)
sentence_audit.to_csv(OUTDIR / "semeval_sentence_audit.csv", index=False)
conflicts.to_csv(OUTDIR / "semeval_conflicting_sentence_labels.csv", index=False)
cross_split.to_csv(OUTDIR / "semeval_cross_split_sentence_overlap.csv", index=False)
cross_domain.to_csv(OUTDIR / "semeval_cross_domain_sentence_overlap.csv", index=False)
flag_prevalence.to_csv(OUTDIR / "semeval_flag_prevalence.csv", index=False)
overlap_matrix.to_csv(OUTDIR / "semeval_overlap_matrix.csv")
overlap_patterns.to_csv(OUTDIR / "semeval_overlap_patterns.csv", index=False)
subgroup_rows.to_csv(OUTDIR / "semeval_subgroup_row_counts_by_domain_split.csv", index=False)
unique_coverage.to_csv(OUTDIR / "semeval_subgroup_unique_sentence_coverage.csv", index=False)
subgroup_class.to_csv(OUTDIR / "semeval_subgroup_class_counts_by_domain_split.csv", index=False)

integrity_summary = {
    "aspect_rows": int(len(df)),
    "unique_texts_global": int(df["text_norm"].nunique()),
    "unique_domain_split_sentences": int(len(sentence_audit)),
    "missing_values": {k: int(v) for k, v in missing_summary.items()},
    "empty_text_rows": empty_text_n,
    "empty_aspect_rows": empty_aspect_n,
    "multi_aspect_domain_split_sentences": multi_n,
    "conflicting_sentiment_domain_split_sentences": conflict_n,
    "within_domain_cross_split_sentence_groups": int(len(cross_split)),
    "cross_domain_sentence_groups": int(len(cross_domain)),
    "skipped_unsupported_label_rows": int(len(skipped_labels)),
}
with open(OUTDIR / "semeval_integrity_summary.json", "w", encoding="utf-8") as f:
    json.dump(integrity_summary, f, indent=2)

with open(OUTDIR / "semeval_role_freeze.json", "w", encoding="utf-8") as f:
    json.dump(role_decision, f, indent=2)

print("Saved authoritative SemEval outputs to:")
print(OUTDIR)
for p in sorted(OUTDIR.iterdir()):
    print(" -", p.name)


Saved authoritative SemEval outputs to:
/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/SemEval2014
 - df_semeval2014_final.parquet
 - semeval_conflicting_sentence_labels.csv
 - semeval_cross_domain_sentence_overlap.csv
 - semeval_cross_split_sentence_overlap.csv
 - semeval_flag_prevalence.csv
 - semeval_integrity_counts.csv
 - semeval_integrity_summary.json
 - semeval_overlap_matrix.csv
 - semeval_overlap_patterns.csv
 - semeval_role_freeze.json
 - semeval_sentence_audit.csv
 - semeval_subgroup_class_counts_by_domain_split.csv
 - semeval_subgroup_row_counts_by_domain_split.csv
 - semeval_subgroup_unique_sentence_coverage.csv


## Cell 14 — Final freeze checklist

In [15]:

print("=" * 72)
print("SEMEVAL-2014 EDA v2 — FREEZE CHECKLIST")
print("=" * 72)
print("Rows:", f"{len(df):,}")
print("Unique global sentence texts:", f"{df['text_norm'].nunique():,}")
print("Conflicting-sentiment sentence records:", f"{conflict_n:,}")
print("Within-domain cross-split sentence groups:", len(cross_split))
print("Frozen TweetEval rules reused:", True)
print("Thresholds tuned on SemEval:", False)
print("Used for training/selection:", False)
print("Role:", role_decision["role_in_thesis"])
print()
print("Coverage summary:")
print(unique_coverage.to_string(index=False))
print()
print("Interpretation:")
print(
    "SemEval-2014 is retained as a domain/task-shift stress test. "
    "It is not treated as a clean same-task external sentiment benchmark."
)


SEMEVAL-2014 EDA v2 — FREEZE CHECKLIST
Rows: 7,694
Unique global sentence texts: 4,444
Conflicting-sentiment sentence records: 604
Within-domain cross-split sentence groups: 2
Frozen TweetEval rules reused: True
Thresholds tuned on SemEval: False
Used for training/selection: False
Role: domain_task_shift_stress_test

Coverage summary:
    domain split           subgroup  unique_sentences  eligible_n_ge_30                                                              interpretation
    laptop  test              other                82              True quantitative coverage available; still subject to task-compatibility limits
    laptop  test reference-unmarked               328              True quantitative coverage available; still subject to task-compatibility limits
    laptop  test        slang-heavy                 1             False                                                descriptive/exploratory only
    laptop train              other               170              True